In [7]:
!pip -q install datasets qdrant-client langchain langchain-community langchain-qdrant transformers accelerate sentence-transformers torch

In [8]:
import re
import math
import json
import torch
import pandas as pd

from typing import List, Dict, Any

from datasets import load_dataset
from qdrant_client import QdrantClient
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from langchain_core.embeddings import Embeddings

from transformers import AutoTokenizer, AutoModel, pipeline

In [32]:
dataset = load_dataset("bearberry/rus_xquadqa")
train_ds = dataset["train"]

In [10]:
def normalize_text_for_dedup(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

all_chunks = []
for row in train_ds:
    for ctx_item in row["context"]:
        chunk = ctx_item["chunk"]
        all_chunks.append(chunk)

unique_chunks = []
seen = set()

for chunk in all_chunks:
    key = normalize_text_for_dedup(chunk)
    if key not in seen:
        seen.add(key)
        unique_chunks.append(chunk)

print("Всего сегментов в датасете:", len(all_chunks))
print("Уникальных сегментов после dedup:", len(unique_chunks))
print("Удалено дублей:", len(all_chunks) - len(unique_chunks))

Всего сегментов в датасете: 6240
Уникальных сегментов после dedup: 1237
Удалено дублей: 5003


In [33]:
documents = [
    Document(
        page_content=chunk,
        metadata={
            "source": "rus_xquadqa_corpus",
            "chunk_id": idx
        }
    )
    for idx, chunk in enumerate(unique_chunks)
]

print("Количество документов для индексации:", len(documents))

Количество документов для индексации: 1237


In [12]:
class MultilingualE5Embeddings(Embeddings):
    def __init__(self, model_name: str = "intfloat/multilingual-e5-large", max_length: int = 512):
        self.model_name = model_name
        self.max_length = max_length

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.model.eval()

    def _average_pool(self, last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
        return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

    def _embed(self, texts: List[str], prefix: str) -> List[List[float]]:
        prepared_texts = [f"{prefix}{text}" for text in texts]

        batch_dict = self.tokenizer(
            prepared_texts,
            max_length=self.max_length,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        batch_dict = {k: v.to(self.device) for k, v in batch_dict.items()}

        with torch.no_grad():
            outputs = self.model(**batch_dict)

        embeddings = self._average_pool(outputs.last_hidden_state, batch_dict["attention_mask"])
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        return embeddings.cpu().tolist()

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self._embed(texts, prefix="passage: ")

    def embed_query(self, text: str) -> List[float]:
        return self._embed([text], prefix="query: ")[0]

In [13]:
embeddings = MultilingualE5Embeddings(
    model_name="intfloat/multilingual-e5-large",
    max_length=512
)

test_vec = embeddings.embed_query("Кто был лидером Пэнтерс по мешкам?")
print("Размерность эмбеддинга:", len(test_vec))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Размерность эмбеддинга: 1024


In [14]:
vector_store = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="rus_xquadqa_collection"
)

print("Индексация завершена.")

Индексация завершена.


In [15]:
def search_in_db(query: str, k: int = 5) -> List[Dict[str, Any]]:
    results = vector_store.similarity_search_with_score(query, k=k)

    output = []
    for doc, score in results:
        output.append({
            "text": doc.page_content,
            "score": float(score),
            "metadata": doc.metadata
        })
    return output

In [16]:
query = train_ds[0]["question"]
retrieved = search_in_db(query, k=5)

print("Вопрос:", query)
print("\nТоп-5 найденных сегментов:\n")

for i, item in enumerate(retrieved, start=1):
    print(f"[{i}] score={item['score']:.4f} | metadata={item['metadata']}")
    print(item["text"])
    print("-" * 100)

Вопрос: Сколько очков уступила защита Пэнтерс?

Топ-5 найденных сегментов:

[1] score=0.8894 | metadata={'source': 'rus_xquadqa_corpus', 'chunk_id': 0, '_id': 'c25752585ada468f9088a6e5ead64461', '_collection_name': 'rus_xquadqa_collection'}
Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по перехватам с 24 и похвасталась четырьмя попаданиями в Пробоул.
----------------------------------------------------------------------------------------------------
[2] score=0.8036 | metadata={'source': 'rus_xquadqa_corpus', 'chunk_id': 5, '_id': '35210819ec3e42e8a25d15df117b84f1', '_collection_name': 'rus_xquadqa_collection'}
Дэвис собрал 51⁄2 мешков, четыре вынужденных потери мяча и четыре перехвата, в то время как Кикли лидировал в команде по блокировкам (118), форсировал две потери мяча и перехватил четыре своих передачи.
----------------------------------------------------------------------------------------------------
[3] score=0.7934 | metadata={'

In [34]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gc
import torch

MODEL_1 = "lmqg/mt5-small-ruquad-qa"
MODEL_2 = "vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa"

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_seq2seq_model(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    model.eval()
    return tokenizer, model

def unload_seq2seq_model(model, tokenizer):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [25]:
def build_rag_prompt(question: str, contexts: list[str]) -> str:
    merged_context = "\n\n".join(contexts)
    prompt = f"question: {question}, context: {merged_context}"
    return prompt

In [26]:
def clean_generated_answer(text: str) -> str:
    text = text.strip()
    text = text.replace("<pad>", "")
    text = text.replace("</s>", "")
    text = text.replace("<extra_id_0>", "")
    text = text.replace("<extra_id_1>", "")
    text = text.replace("<extra_id_2>", "")
    text = text.replace("<extra_id_3>", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def generate_answer_with_rag(question: str, model, tokenizer, k: int = 3, max_new_tokens: int = 32) -> dict:
    retrieved = search_in_db(question, k=k)
    contexts = [item["text"] for item in retrieved]

    prompt = build_rag_prompt(question, contexts)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

    generated = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    generated = clean_generated_answer(generated)

    return {
        "question": question,
        "answer": generated,
        "contexts": contexts,
        "retrieved": retrieved,
        "prompt": prompt
    }

In [27]:
test_question = train_ds[0]["question"]

tokenizer_1, model_1 = load_seq2seq_model(MODEL_1)
result_1 = generate_answer_with_rag(test_question, model_1, tokenizer_1, k=3)
unload_seq2seq_model(model_1, tokenizer_1)

tokenizer_2, model_2 = load_seq2seq_model(MODEL_2)
result_2 = generate_answer_with_rag(test_question, model_2, tokenizer_2, k=3)
unload_seq2seq_model(model_2, tokenizer_2)

print("Вопрос:", test_question)

print("\nОтвет модели 1:")
print(result_1["answer"])

print("\nОтвет модели 2:")
print(result_2["answer"])

print("\nЭталон:")
print(train_ds[0]["answers"])

config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/191 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/545M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Вопрос: Сколько очков уступила защита Пэнтерс?

Ответ модели 1:
308 очков

Ответ модели 2:
308 очков

Эталон:
['308']


In [28]:
def normalize_answer_text(text: str) -> str:
    text = text.lower().strip()
    text = text.replace("ё", "е")
    text = re.sub(r"[\"'«»„“”]", "", text)
    text = re.sub(r"[^\w\s.-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def is_match(prediction: str, gold_normalized_answers: List[str]) -> bool:
    pred_norm = normalize_answer_text(prediction)
    gold_norm = [normalize_answer_text(ans) for ans in gold_normalized_answers]

    if pred_norm in gold_norm:
        return True

    for gold in gold_norm:
        if pred_norm == gold:
            return True
        if pred_norm in gold:
            return True
        if gold in pred_norm:
            return True

    return False

In [29]:
top_n = 10
rows = [train_ds[i] for i in range(top_n)]

results_table = []

tokenizer_1, model_1 = load_seq2seq_model(MODEL_1)

model_1_answers = []
for row in rows:
    question = row["question"]
    rag_1 = generate_answer_with_rag(question, model_1, tokenizer_1, k=3)
    model_1_answers.append(rag_1["answer"])

unload_seq2seq_model(model_1, tokenizer_1)

tokenizer_2, model_2 = load_seq2seq_model(MODEL_2)

model_2_answers = []
for row in rows:
    question = row["question"]
    rag_2 = generate_answer_with_rag(question, model_2, tokenizer_2, k=3)
    model_2_answers.append(rag_2["answer"])

unload_seq2seq_model(model_2, tokenizer_2)

for idx, row in enumerate(rows, start=1):
    question = row["question"]
    gold_answers = row["answers"]
    gold_normalized_answers = row["normalized_answers"]

    pred_1 = model_1_answers[idx - 1]
    pred_2 = model_2_answers[idx - 1]

    match_1 = is_match(pred_1, gold_normalized_answers)
    match_2 = is_match(pred_2, gold_normalized_answers)

    results_table.append({
        "n": idx,
        "question": question,
        "gold_answers": gold_answers,
        "gold_normalized_answers": gold_normalized_answers,
        "model_1_name": MODEL_1,
        "model_1_answer": pred_1,
        "model_1_match": match_1,
        "model_2_name": MODEL_2,
        "model_2_answer": pred_2,
        "model_2_match": match_2
    })

results_df = pd.DataFrame(results_table)
results_df

Loading weights:   0%|          | 0/191 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/191 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


,n,question,gold_answers,gold_normalized_answers,model_1_name,model_1_answer,model_1_match,model_2_name,model_2_answer,model_2_match
0,1,Сколько очков уступила защита Пэнтерс?,[308],[308],lmqg/mt5-small-ruquad-qa,308 очков,True,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,308 очков,True
1,2,Сколько мешков за карьеру было у Джареда Аллена?,[136],[136],lmqg/mt5-small-ruquad-qa,5 мешков,False,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,5 мешков,False
2,3,Сколько блокировок записал на свой счет Люк Ки...,[118],[118],lmqg/mt5-small-ruquad-qa,две потери мяча,False,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,118,True
3,4,Сколько мячей перехватил Джош Норман?,"[4, четыре]","[четыре, 4]",lmqg/mt5-small-ruquad-qa,четыре,True,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,четыре своих передачи,True
4,5,Кто больше записал на свой счет мешков в коман...,[Кейван Шорт],[кейван шорты],lmqg/mt5-small-ruquad-qa,Дифенсив тэкл Пробоула,False,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,Дэвис,False
5,6,Сколько перехватов приписывают защите Пэнтерс ...,[24],[24],lmqg/mt5-small-ruquad-qa,7 перехватов,False,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,308 очков,False
6,7,Кто был лидером Пэнтерс по мешкам?,[Кейван Шорт],[кейван шорты],lmqg/mt5-small-ruquad-qa,Пробоула,False,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,ди-энда-ветерана Джареда Аллена,False
7,8,Сколько игроков защиты Пэнтерс было выбрано дл...,"[4, четыре]","[четыре, 4]",lmqg/mt5-small-ruquad-qa,308 очков,False,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,308 очков,False
8,9,Сколько вынужденных потерь мяча имел Томас Дэвис?,"[4, четыре]","[четыре, 4]",lmqg/mt5-small-ruquad-qa,четыре,True,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,четыре вынужденных потери мяча,True
9,10,У какого игрока было больше всего перехватов в...,[Курт Колеман],[курт колеман],lmqg/mt5-small-ruquad-qa,Джош Норман,False,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,Джош Норман,False


In [30]:
acc_model_1 = results_df["model_1_match"].mean()
acc_model_2 = results_df["model_2_match"].mean()

summary_df = pd.DataFrame({
    "model": [MODEL_1, MODEL_2],
    "top10_accuracy": [acc_model_1, acc_model_2]
})

summary_df

,model,top10_accuracy
0,lmqg/mt5-small-ruquad-qa,0.3
1,vocabtrimmer/mt5-small-trimmed-ru-90000-ruquad-qa,0.4
